# 第2章 交通监测数据整编

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch02-audit-v1`  
**必做：** 时间索引审计；同小时一致性核查与去重  
**对象：** 小时交通量：辆/小时  
**样本：** 48204原始行，40575个本地小时标签  
**划分：** 描述性整编，不构造训练与测试  
**比较：** 先核对同小时交通量一致，再比较原始行加权与唯一小时；缺测不填0

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 读取原表与观测时刻
请解释：同小时多行是否就是重复车辆？此处是一份小时交通量表，不是逐车记录。


In [ ]:
raw = pd.read_csv(ROOT/'chapters/data/ch02_raw.csv', keep_default_na=False)
raw['time'] = pd.to_datetime(raw['date_time'])
display(raw.head())
print('Raw rows:', len(raw), 'Distinct hours:', raw['time'].nunique())


## 2. 冲突核查先于去重
修改检查字段时，解释为什么不能任意保留第一条不同值。时间缺口与字段空值要分别检查。


In [ ]:
conflicts = raw.groupby('time').traffic_volume.nunique()
assert not (conflicts > 1).any(), 'Different traffic volumes share a timestamp; investigate before deduplication'
audit = []
for year, part in raw.groupby(raw.time.dt.year):
    start_time, end_time = part.time.min(), part.time.max()
    skeleton = pd.date_range(start_time, end_time, freq='h')
    audit.append([int(year), len(part), part.time.nunique(), len(part)-part.time.nunique(), len(skeleton.difference(part.time))])
quality = pd.DataFrame(audit, columns=['year','raw_rows','unique_hours','extra_rows','missing_labels_within_coverage'])
display(quality)
csv_file(ROOT/'outputs/ch02/quality_audit.csv', quality.columns, quality.values)


## 3. 生成整编结果并比较日变化曲线
下面不将缺测补0。修改统计年份，说明你在比较哪些日期的均值。


In [ ]:
clean = raw.drop_duplicates('time').sort_values('time')
assert len(clean) == 40575
csv_file(ROOT/'outputs/ch02/ch02_clean.csv', ['local_time','volume'], zip(clean.time.dt.strftime('%Y-%m-%dT%H:%M'),clean.traffic_volume))
YEAR = 2017  # 可修改；须与图的解释一致
weighted = raw[raw.time.dt.year == YEAR].groupby(raw.time.dt.hour).traffic_volume.mean()
unique = clean[clean.time.dt.year == YEAR].groupby(clean.time.dt.hour).traffic_volume.mean()
plt.figure(figsize=(9,4)); plt.plot(weighted.index,weighted,label='Raw-row weighted'); plt.plot(unique.index,unique,label='Unique hours')
plt.xlabel('Local hour');plt.ylabel('Vehicles/hour');plt.legend();save_plot(ROOT,2,'hourly_profile')


## 4. 写出数据接收结论
选择一处曲线差异和一段缺口，说明可能原因与所需核查证据。


In [ ]:
report(ROOT,2,'道路监测数据接收说明',{'质量审计':quality.to_string(index=False),'整编小时数':len(clean),'比较年份':YEAR},['去重的证据是什么？','最大差异时段如何解释？','哪些缺口仍需向数据提供方核查？'])
